In [2]:
#!/usr/bin/env python3
"""
Bulk download Stormwatch rain increment data.

Input: CSV with columns like:
  site_id, device_id, site, device, Sensor Title, sensor_url (any extra columns are ok)

Auth: requires WEBAPP_SESSION cookie from a logged-in browser session.

Output:
  out/
    metadata/
      sensors_meta.csv
      rain_increment_sensors.csv
    data/
      site{site_id}_dev{device_id}_bin3600.csv
"""

from __future__ import annotations

import argparse
import csv
import json
import re
import time
from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import requests


API_URL = "https://www.stormwatch.com/api/graph/flot/"
Default_dir="/mnt/12TB/Sujan/Spatial_correlation/Codes/dependent_files/stormwatch_sensor_info.csv"
default_out="/mnt/12TB/Sujan/Spatial_correlation/Codes/SW_Downloads/"
cookie="uc9l684mcrjhs3kj6elft51slu"

@dataclass
class SensorRow:
    site_id: int
    device_id: int
    site_uuid: Optional[str] = None
    device_uuid: Optional[str] = None
    sensor_title: Optional[str] = None
    sensor_url: Optional[str] = None


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser()
    p.add_argument("--in-csv", default=str(Default_dir), help="Input CSV exported from your sensor table")
    p.add_argument("--out-dir", default=str(default_out), help="Output directory")
    p.add_argument("--cookie", default=str(cookie), help="WEBAPP_SESSION cookie value")
    p.add_argument("--start", default="2000-01-01 00:00:00", help="Start datetime (local to time_zone)")
    p.add_argument("--end", default="2026-02-01 23:59:59", help="End datetime (local to time_zone)")
    p.add_argument("--bin", type=int, default=3600, help="Aggregation seconds. 3600=hourly")
    p.add_argument("--time-zone", default="US/Central", help="Stormwatch time_zone parameter")
    p.add_argument("--chunk-days", type=int, default=31, help="Download in chunks to avoid huge responses")
    p.add_argument("--sleep", type=float, default=0.3, help="Sleep between requests (seconds)")
    p.add_argument("--timeout", type=int, default=60, help="Request timeout seconds")
    return p.parse_args()


def load_sensor_table(path: Path) -> List[SensorRow]:
    df = pd.read_csv(path)

    # normalize column names
    cols = {c.strip(): c for c in df.columns}

    def get_col(*names: str) -> Optional[str]:
        for n in names:
            if n in cols:
                return cols[n]
        # try case-insensitive match
        lower = {c.lower(): c for c in df.columns}
        for n in names:
            if n.lower() in lower:
                return lower[n.lower()]
        return None

    c_site_id = get_col("site_id", "Site ID")
    c_device_id = get_col("device_id", "Device ID")
    if not c_site_id or not c_device_id:
        raise ValueError("Input CSV must contain columns site_id and device_id (case-insensitive match is ok).")

    c_site_uuid = get_col("site", "site_uuid", "Site")
    c_device_uuid = get_col("device", "device_uuid", "Device")
    c_title = get_col("Sensor Title", "sensor_title", "title")
    c_url = get_col("sensor_url", "url")

    rows: List[SensorRow] = []
    for _, r in df.iterrows():
        rows.append(
            SensorRow(
                site_id=int(r[c_site_id]),
                device_id=int(r[c_device_id]),
                site_uuid=str(r[c_site_uuid]) if c_site_uuid and pd.notna(r[c_site_uuid]) else None,
                device_uuid=str(r[c_device_uuid]) if c_device_uuid and pd.notna(r[c_device_uuid]) else None,
                sensor_title=str(r[c_title]) if c_title and pd.notna(r[c_title]) else None,
                sensor_url=str(r[c_url]) if c_url and pd.notna(r[c_url]) else None,
            )
        )
    return rows


def make_session(webapp_session_value: str) -> requests.Session:
    s = requests.Session()
    # Only the session cookie is usually needed
    s.cookies.set("WEBAPP_SESSION", webapp_session_value, domain="www.stormwatch.com", path="/")
    return s


def call_sensor_details(
    s: requests.Session,
    site_id: int,
    device_id: int,
    start_dt: str,
    end_dt: str,
    bin_sec: int,
    tz: str,
    timeout: int = 60,
) -> Any:
    params = {
        "method": "sensor_details",
        "site_id": site_id,
        "device_id": device_id,
        "site": site_id,      # yes, Stormwatch repeats these
        "device": device_id,
        "data_start": start_dt,
        "data_end": end_dt,
        "forecast_time": "",
        "range": "",
        "bin": bin_sec,
        "time_zone": tz,
        "show_assoc_sensors": 1,
        # "_": int(time.time() * 1000),  # cache buster; optional
    }

    headers = {
        "Accept": "application/json, text/javascript, */*; q=0.01",
        "X-Requested-With": "XMLHttpRequest",
        "Referer": "https://www.stormwatch.com/",
    }

    r = s.get(API_URL, params=params, headers=headers, timeout=timeout)
    # If session is invalid you often get HTML/login or 302 behavior; fail loudly
    ct = r.headers.get("content-type", "")
    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
    if "application/json" not in ct:
        raise RuntimeError(f"Expected JSON but got content-type={ct}. Response head: {r.text[:200]}")
    return r.json()


def extract_timeseries(payload: Any) -> Tuple[Dict[str, Any], Optional[List[List[Any]]]]:
    """
    Returns (meta, series) where series is [[ts_ms, value], ...] or None if not found.
    Meta is the main object we used.
    """
    obj = payload[0] if isinstance(payload, list) and payload else payload
    if not isinstance(obj, dict):
        return ({"_raw_type": str(type(obj))}, None)

    # Common pattern: {"data": [[ms,val],...], "label": "...", "units":"in", ...}
    if "data" in obj and isinstance(obj["data"], list):
        return (obj, obj["data"])

    # Some flot payloads nest series
    for key in ("series", "datasets", "sensors"):
        if key in obj and isinstance(obj[key], list) and obj[key]:
            cand = obj[key][0]
            if isinstance(cand, dict) and "data" in cand and isinstance(cand["data"], list):
                return (cand, cand["data"])

    return (obj, None)


def looks_like_rain_increment(meta: Dict[str, Any], fallback_title: Optional[str]) -> bool:
    """
    Heuristic detection:
    - label/title contains 'rain' and 'increment' or similar
    - OR units look like inches/mm and label contains rain
    """
    label = str(meta.get("label") or meta.get("title") or meta.get("name") or fallback_title or "").lower()
    units = str(meta.get("units") or "").lower()

    # strong match
    if "rain" in label and ("increment" in label or "inc" in label):
        return True

    # weaker match
    if "rain" in label and units in {"in", "inch", "inches", "mm"}:
        return True

    return False


def dt_range_chunks(start: datetime, end: datetime, chunk_days: int) -> List[Tuple[datetime, datetime]]:
    out = []
    cur = start
    step = timedelta(days=chunk_days)
    while cur <= end:
        nxt = min(cur + step - timedelta(seconds=1), end)
        out.append((cur, nxt))
        cur = nxt + timedelta(seconds=1)
    return out


def to_stormwatch_dt(dt: datetime) -> str:
    return dt.strftime("%Y-%m-%d %H:%M:%S")


def main() -> None:
    args = parse_args()

    in_csv = Path(args.in_csv)
    out_dir = Path(args.out_dir)
    data_dir = out_dir / "data"
    meta_dir = out_dir / "metadata"
    data_dir.mkdir(parents=True, exist_ok=True)
    meta_dir.mkdir(parents=True, exist_ok=True)

    sensors = load_sensor_table(in_csv)
    s = make_session(args.cookie)

    start_dt = datetime.strptime(args.start, "%Y-%m-%d %H:%M:%S")
    end_dt = datetime.strptime(args.end, "%Y-%m-%d %H:%M:%S")

    meta_rows: List[Dict[str, Any]] = []
    rain_sensors: List[SensorRow] = []

    # 1) Scrape meta + detect rain-increment
    for i, sr in enumerate(sensors, 1):
        try:
            # small probe range to keep it light (1 day)
            probe_start = to_stormwatch_dt(start_dt)
            probe_end = to_stormwatch_dt(min(start_dt + timedelta(days=1) - timedelta(seconds=1), end_dt))

            payload = call_sensor_details(
                s, sr.site_id, sr.device_id, probe_start, probe_end, args.bin, args.time_zone, args.timeout
            )
            meta, series = extract_timeseries(payload)

            count = len(series) if series else 0
            units = meta.get("units", None)
            label = meta.get("label", None)

            is_rain = looks_like_rain_increment(meta, sr.sensor_title) and count > 0

            meta_rows.append(
                {
                    "site_id": sr.site_id,
                    "device_id": sr.device_id,
                    "site_uuid": sr.site_uuid,
                    "device_uuid": sr.device_uuid,
                    "input_title": sr.sensor_title,
                    "api_label": label,
                    "units": units,
                    "probe_count": count,
                    "is_rain_increment": bool(is_rain),
                }
            )

            if is_rain:
                rain_sensors.append(sr)

        except Exception as e:
            meta_rows.append(
                {
                    "site_id": sr.site_id,
                    "device_id": sr.device_id,
                    "site_uuid": sr.site_uuid,
                    "device_uuid": sr.device_uuid,
                    "input_title": sr.sensor_title,
                    "api_label": None,
                    "units": None,
                    "probe_count": 0,
                    "is_rain_increment": False,
                    "error": str(e),
                }
            )

        time.sleep(args.sleep)

    meta_df = pd.DataFrame(meta_rows)
    meta_df.to_csv(meta_dir / "sensors_meta.csv", index=False)

    rain_df = meta_df[meta_df["is_rain_increment"] == True].copy()
    rain_df.to_csv(meta_dir / "rain_increment_sensors.csv", index=False)

    print(f"Found {len(rain_sensors)} rain-increment sensors (see {meta_dir/'rain_increment_sensors.csv'}).")

    # 2) Download full data for rain sensors (chunked), write one CSV per sensor
    chunks = dt_range_chunks(start_dt, end_dt, args.chunk_days)

    for sr in rain_sensors:
        out_csv = data_dir / f"site{sr.site_id}_dev{sr.device_id}_bin{args.bin}.csv"
        rows_out: List[Tuple[int, Any]] = []

        for c0, c1 in chunks:
            try:
                payload = call_sensor_details(
                    s,
                    sr.site_id,
                    sr.device_id,
                    to_stormwatch_dt(c0),
                    to_stormwatch_dt(c1),
                    args.bin,
                    args.time_zone,
                    args.timeout,
                )
                meta, series = extract_timeseries(payload)
                if series:
                    rows_out.extend((int(x[0]), x[1]) for x in series if isinstance(x, list) and len(x) >= 2)
            except Exception as e:
                print(f"[WARN] site{sr.site_id} dev{sr.device_id} chunk {c0}..{c1}: {e}")

            time.sleep(args.sleep)

        if not rows_out:
            print(f"[WARN] No data written for site{sr.site_id} dev{sr.device_id}")
            continue

        # Deduplicate & sort
        df = pd.DataFrame(rows_out, columns=["ts_ms", "value"])
        df = df.dropna(subset=["ts_ms"]).drop_duplicates(subset=["ts_ms"]).sort_values("ts_ms")

        # Add time columns (UTC + Central) for hydrology-friendly handling
        df["time_utc"] = pd.to_datetime(df["ts_ms"], unit="ms", utc=True)
        df["time_central"] = df["time_utc"].dt.tz_convert(args.time_zone)

        df.to_csv(out_csv, index=False)
        print(f"Wrote {out_csv} ({len(df)} rows)")

    print("Done.")


if __name__ == "__main__":
    main()


usage: ipykernel_launcher.py [-h] [--in-csv IN_CSV] [--out-dir OUT_DIR]
                             [--cookie COOKIE] [--start START] [--end END]
                             [--bin BIN] [--time-zone TIME_ZONE]
                             [--chunk-days CHUNK_DAYS] [--sleep SLEEP]
                             [--timeout TIMEOUT]
ipykernel_launcher.py: error: unrecognized arguments: -f /home/smncd/.local/share/jupyter/runtime/kernel-08670deb-f2fe-41a6-9fc8-26f78517615d.json


SystemExit: 2

/mnt/12TB/Sujan/conda_envs/spa_cor/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [3]:
%tb


SystemExit: 2